# 5.1 LSTM Forecasting

這份 Notebook 示範如何把單變量時間序列轉成 sliding window，並使用 LSTM 預測下一個時間點。


## 1. 載入套件


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

tf.keras.utils.set_random_seed(42)
from sklearn.metrics import mean_absolute_error, mean_squared_error


## 2. 建立合成時間序列

序列包含趨勢、週期與雜訊，用來模擬常見的流量、銷售或感測器資料。


In [ ]:
rng = np.random.default_rng(42)
n_points = 900
time_index = np.arange(n_points)
series = (
    0.004 * time_index
    + np.sin(2 * np.pi * time_index / 24)
    + 0.45 * np.sin(2 * np.pi * time_index / 96)
    + rng.normal(0, 0.12, n_points)
).astype('float32')

plt.figure(figsize=(10, 3))
plt.plot(time_index, series)
plt.title('Synthetic time series')
plt.xlabel('time')
plt.ylabel('value')
plt.show()


## 3. 建立 sliding window

使用過去 `WINDOW_SIZE` 個時間點預測下一個時間點。


In [ ]:
WINDOW_SIZE = 24

def make_windows(values, window_size):
    x, y = [], []
    for i in range(len(values) - window_size):
        x.append(values[i:i + window_size])
        y.append(values[i + window_size])
    x = np.array(x, dtype='float32')[..., np.newaxis]
    y = np.array(y, dtype='float32')
    return x, y

X, y = make_windows(series, WINDOW_SIZE)
print('X:', X.shape, 'y:', y.shape)


## 4. 依時間順序切分與標準化

時間序列預測不使用隨機切分。標準化只使用訓練集統計量。


In [ ]:
n = len(X)
train_end = int(n * 0.7)
valid_end = int(n * 0.85)

x_train, y_train = X[:train_end], y[:train_end]
x_valid, y_valid = X[train_end:valid_end], y[train_end:valid_end]
x_test, y_test = X[valid_end:], y[valid_end:]

mean = x_train.mean()
std = x_train.std() + 1e-8
x_train_s = (x_train - mean) / std
x_valid_s = (x_valid - mean) / std
x_test_s = (x_test - mean) / std
y_train_s = (y_train - mean) / std
y_valid_s = (y_valid - mean) / std
y_test_s = (y_test - mean) / std

print(x_train_s.shape, x_valid_s.shape, x_test_s.shape)


## 5. 建立 LSTM forecasting model


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(WINDOW_SIZE, 1)),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1),
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()


## 6. 訓練模型


In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(
    x_train_s, y_train_s,
    validation_data=(x_valid_s, y_valid_s),
    epochs=35,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0,
)
pd.DataFrame(history.history)[['loss', 'val_loss']].plot(figsize=(8, 4), title='Training Curve')
plt.show()
print('epochs:', len(history.history['loss']))


## 7. 評估預測結果


In [ ]:
y_pred_s = model.predict(x_test_s, verbose=0).ravel()
y_pred = y_pred_s * std + mean
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
pd.DataFrame({'metric': ['MAE', 'RMSE'], 'value': [mae, rmse]})


## 8. 視覺化實際值與預測值


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(y_test[:160], label='actual')
plt.plot(y_pred[:160], label='predicted')
plt.title('LSTM forecasting on test set')
plt.legend()
plt.show()


## 9. 如何套用自己的資料

將自己的時間欄位排序後，選定目標欄位與 `WINDOW_SIZE`。如果有多個特徵，`X` 的最後一維會變成特徵數。切分時要保留時間順序，標準化也只能 fit 訓練集。


## 10. 小結

LSTM forecasting 的核心流程是 sliding window、時間切分、訓練集標準化與預測曲線檢查。模型可以替換，但資料流程要先正確。
